# Notebook A - DermLIP embedding & 7-point concept cache (Derm7pt)

Builds the **frozen-feature cache** that every later RL / faithfulness / fairness experiment reads.
Encoder is frozen, so this runs on a Kaggle free GPU (P100 or T4) in a few minutes; afterwards you iterate on cached vectors with almost no GPU.

**Kaggle setup**
1. Settings -> Accelerator -> **GPU**.
2. Settings -> **Internet ON** (needed once to download DermLIP from Hugging Face).
3. Add Input -> attach a **Derm7pt** dataset that contains `meta.csv`, the split index csvs (`train_indexes.csv`, `valid_indexes.csv`, `test_indexes.csv`), and the `images/` folder. (Derm7pt: Kawahara et al.; register at derm.cs.sfu.ca, or use a Kaggle mirror.)
4. Run all.

**Output** -> `/kaggle/working/derm7pt_dermlip_cache/` (`features.npz`, `meta.parquet`, `manifest.json`). Save a Version, then create a Kaggle Dataset from the output and feed it to Notebook B.

In [ ]:
# --- 1. Install ---
!pip install -q open_clip_torch
print('open_clip installed')

In [ ]:
# --- 2. Imports & config ---
import os, glob, json, time
import numpy as np
import pandas as pd
import torch
import open_clip
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DEV_TYPE = 'cuda' if DEVICE == 'cuda' else 'cpu'
MODEL_NAME = 'hf-hub:redlessone/DermLIP_ViT-B-16'
BATCH_SIZE = 32
DERM7PT_ROOT = ''            # leave empty to auto-search /kaggle/input
OUT_DIR = '/kaggle/working/derm7pt_dermlip_cache'
print('Device:', DEVICE)

In [ ]:
# --- 3. Load DermLIP (frozen) ---
print('Loading DermLIP ...')
model, _, preprocess = open_clip.create_model_and_transforms(MODEL_NAME)
tokenizer = open_clip.get_tokenizer(MODEL_NAME)
model = model.to(DEVICE).eval()
for p in model.parameters():
    p.requires_grad = False
n_params = sum(p.numel() for p in model.parameters())
print('DermLIP loaded. Params (M):', round(n_params / 1e6, 1))

In [ ]:
# --- 4. Locate & load Derm7pt metadata + splits ---
search_roots = [DERM7PT_ROOT] if DERM7PT_ROOT else ['/kaggle/input']

def find_one(name, roots):
    for r in roots:
        if not r:
            continue
        hits = sorted(glob.glob(os.path.join(r, '**', name), recursive=True))
        if hits:
            return hits[0]
    return None

meta_path = find_one('meta.csv', search_roots)
if meta_path is None:
    raise FileNotFoundError('Could not find meta.csv. Attach the Derm7pt dataset (Add Input) or set DERM7PT_ROOT.')
print('meta.csv:', meta_path)

meta_dir = os.path.dirname(meta_path)
cand_images = [os.path.join(os.path.dirname(meta_dir), 'images'), os.path.join(meta_dir, 'images')]
IMAGES_DIR = next((c for c in cand_images if os.path.isdir(c)), None)
if IMAGES_DIR is None:
    for r in search_roots:
        hits = [d for d in glob.glob(os.path.join(r, '**', 'images'), recursive=True) if os.path.isdir(d)]
        if hits:
            IMAGES_DIR = hits[0]
            break
print('images dir:', IMAGES_DIR)

df = pd.read_csv(meta_path).reset_index(drop=True)
print('cases:', len(df))
print('columns:', list(df.columns))

split_arr = np.array(['unused'] * len(df), dtype=object)
split_files = {'train': 'train_indexes.csv', 'val': 'valid_indexes.csv', 'test': 'test_indexes.csv'}
for sname, fname in split_files.items():
    p = find_one(fname, search_roots)
    if p is None:
        print('WARN: split file not found:', fname)
        continue
    idx = pd.read_csv(p).iloc[:, 0].astype(int).tolist()
    idx = [i for i in idx if 0 <= i < len(df)]
    for i in idx:
        split_arr[i] = sname
print('split counts:', {sn: int((split_arr == sn).sum()) for sn in ['train', 'val', 'test', 'unused']})

In [ ]:
# --- 5. Inspect label vocab & build 7-point ground truth ---
# Print the actual category strings so you can adjust the mapping below if your
# Derm7pt copy uses different spellings.
crit_cols = ['pigment_network', 'blue_whitish_veil', 'vascular_structures', 'pigmentation', 'streaks', 'dots_and_globules', 'regression_structures']
for c in crit_cols:
    if c in df.columns:
        print(c, '->', sorted([str(v) for v in df[c].dropna().unique()]))
    else:
        print('MISSING column:', c)

def s(row, col):
    return str(row.get(col, '')).strip().lower()

# 7-point checklist: 3 major criteria (2 pts) + 4 minor criteria (1 pt); score >= 3 -> suspicious
CONCEPTS = ['atypical_pigment_network', 'blue_whitish_veil', 'atypical_vascular', 'irregular_streaks', 'irregular_pigmentation', 'irregular_dots_globules', 'regression']
WEIGHTS = np.array([2, 2, 2, 1, 1, 1, 1], dtype=np.float32)

def gt_concept_vector(row):
    apn = s(row, 'pigment_network') == 'atypical'
    bwv = s(row, 'blue_whitish_veil') == 'present'
    avasc = s(row, 'vascular_structures') in ['dotted', 'linear irregular']
    strk = s(row, 'streaks') == 'irregular'
    pig = s(row, 'pigmentation') in ['diffuse irregular', 'localized irregular']
    dots = s(row, 'dots_and_globules') == 'irregular'
    regr = s(row, 'regression_structures') not in ['absent', '', 'nan']
    return np.array([apn, bwv, avasc, strk, pig, dots, regr], dtype=np.float32)

gt_concepts = np.stack([gt_concept_vector(r) for _, r in df.iterrows()])
gt_score = (gt_concepts * WEIGHTS).sum(axis=1)
mel_suspicious = (gt_score >= 3).astype(np.int64)

diag = df['diagnosis'].astype(str).str.lower() if 'diagnosis' in df.columns else pd.Series([''] * len(df))
mel_label = diag.str.contains('melanoma').astype(np.int64).values
print('melanoma cases:', int(mel_label.sum()), 'of', len(df))
print('7pt-suspicious cases (score>=3):', int(mel_suspicious.sum()))

In [ ]:
# --- 6. Concept & diagnosis text prompts (DermLIP zero-shot concept annotation) ---
# Each concept = a (present, absent) prompt pair; score = softmax probability of 'present'.
CONCEPT_PROMPTS = {
    'atypical_pigment_network': ('dermoscopy of a skin lesion with an atypical pigment network', 'dermoscopy of a skin lesion with a typical pigment network'),
    'blue_whitish_veil': ('dermoscopy of a skin lesion with a blue-whitish veil', 'dermoscopy of a skin lesion without a blue-whitish veil'),
    'atypical_vascular': ('dermoscopy of a skin lesion with atypical vascular structures', 'dermoscopy of a skin lesion with regular vascular structures'),
    'irregular_streaks': ('dermoscopy of a skin lesion with irregular streaks', 'dermoscopy of a skin lesion without irregular streaks'),
    'irregular_pigmentation': ('dermoscopy of a skin lesion with irregular pigmentation', 'dermoscopy of a skin lesion with regular pigmentation'),
    'irregular_dots_globules': ('dermoscopy of a skin lesion with irregular dots and globules', 'dermoscopy of a skin lesion with regular dots and globules'),
    'regression': ('dermoscopy of a skin lesion with regression structures', 'dermoscopy of a skin lesion without regression structures'),
}
DIAG_PROMPTS = ('dermoscopy of a malignant melanoma', 'dermoscopy of a benign skin lesion')

In [ ]:
# --- 7. Encode / score helpers ---
def encode_images(paths):
    n = len(paths)
    feats, order = [], []
    i = 0
    while i < n:
        batch = paths[i:i + BATCH_SIZE]
        tens, idxs = [], []
        for j, p in enumerate(batch):
            try:
                im = Image.open(p).convert('RGB')
                tens.append(preprocess(im))
                idxs.append(i + j)
            except Exception:
                pass
        if tens:
            x = torch.stack(tens).to(DEVICE)
            with torch.no_grad(), torch.autocast(device_type=DEV_TYPE, enabled=(DEV_TYPE == 'cuda')):
                f = model.encode_image(x)
                f = f / f.norm(dim=-1, keepdim=True)
            feats.append(f.float().cpu().numpy())
            order.extend(idxs)
        i += BATCH_SIZE
    feats = np.concatenate(feats, axis=0) if feats else np.zeros((0, 512), np.float32)
    d = feats.shape[1] if feats.shape[0] else 512
    emb = np.zeros((n, d), dtype=np.float32)
    mask = np.zeros(n, dtype=bool)
    for k, idx in enumerate(order):
        emb[idx] = feats[k]
        mask[idx] = True
    return emb, mask

def encode_text(prompts):
    tok = tokenizer(list(prompts)).to(DEVICE)
    with torch.no_grad(), torch.autocast(device_type=DEV_TYPE, enabled=(DEV_TYPE == 'cuda')):
        t = model.encode_text(tok)
        t = t / t.norm(dim=-1, keepdim=True)
    return t.float().cpu().numpy()

def softmax2(a, b):
    z = np.stack([a, b], axis=1) * 100.0
    z = z - z.max(axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

def concept_scores(emb, mask):
    names = list(CONCEPT_PROMPTS.keys())
    present = encode_text([CONCEPT_PROMPTS[k][0] for k in names])
    absent = encode_text([CONCEPT_PROMPTS[k][1] for k in names])
    out = np.zeros((emb.shape[0], len(names)), dtype=np.float32)
    for c in range(len(names)):
        p = softmax2(absent[c] @ emb.T, present[c] @ emb.T)
        out[:, c] = p[:, 1]
    out[~mask] = np.nan
    return out, names

def diag_melanoma_prob(emb, mask):
    t = encode_text(list(DIAG_PROMPTS))
    p = softmax2(t[1] @ emb.T, t[0] @ emb.T)
    out = p[:, 1]
    out[~mask] = np.nan
    return out

In [ ]:
# --- 8. Run extraction (dermoscopic + clinical) ---
def build_paths(col):
    if col not in df.columns or IMAGES_DIR is None:
        return [None] * len(df)
    return [os.path.join(IMAGES_DIR, str(v).strip()) for v in df[col].tolist()]

derm_paths = build_paths('derm')
clinic_paths = build_paths('clinic')

t0 = time.time()
print('Encoding dermoscopic images ...')
emb_derm, mask_derm = encode_images(derm_paths)
print('  ok:', int(mask_derm.sum()), '/', len(derm_paths))
print('Encoding clinical images ...')
emb_clinic, mask_clinic = encode_images(clinic_paths)
print('  ok:', int(mask_clinic.sum()), '/', len(clinic_paths))
print('Scoring concepts ...')
cs_derm, concept_names = concept_scores(emb_derm, mask_derm)
cs_clinic, _ = concept_scores(emb_clinic, mask_clinic)
mel_prob_derm = diag_melanoma_prob(emb_derm, mask_derm)
print('Done in', round(time.time() - t0, 1), 's. Embedding dim:', emb_derm.shape[1])

In [ ]:
# --- 9. Sanity checks ---
# (a) DermLIP is discriminative; (b) concept prompts track the real criteria;
# (c) the symbolic 7pt rule is itself predictive (sanity for the faithfulness verifier).
from sklearn.metrics import roc_auc_score

test = (split_arr == 'test')
if test.sum() == 0:
    test = np.ones(len(df), dtype=bool)
    print('(no test split found -> reporting on all cases)')

def safe_auc(y, x):
    m = ~np.isnan(x)
    y, x = y[m], x[m]
    if len(np.unique(y)) < 2:
        return float('nan')
    return roc_auc_score(y, x)

print('Zero-shot melanoma AUROC (DermLIP, derm, test):', round(safe_auc(mel_label[test], mel_prob_derm[test]), 3))
print('Rule-based 7pt AUROC (GT score vs GT melanoma):', round(safe_auc(mel_label, gt_score.astype(float)), 3))
print()
print('Per-concept AUROC -- DermLIP concept score vs Derm7pt ground-truth criterion (derm):')
for c, name in enumerate(concept_names):
    print('  {0:26s} AUROC={1}'.format(name, round(safe_auc(gt_concepts[:, c], cs_derm[:, c]), 3)))

In [ ]:
# --- 10. Save cache ---
os.makedirs(OUT_DIR, exist_ok=True)
np.savez_compressed(
    os.path.join(OUT_DIR, 'features.npz'),
    emb_derm=emb_derm, mask_derm=mask_derm,
    emb_clinic=emb_clinic, mask_clinic=mask_clinic,
    concept_scores_derm=cs_derm, concept_scores_clinic=cs_clinic,
    gt_concepts=gt_concepts, gt_score=gt_score,
    mel_suspicious=mel_suspicious, mel_label=mel_label,
    mel_prob_derm=mel_prob_derm, split=split_arr.astype('U8'),
)

keep = ['case_num', 'diagnosis', 'sex', 'location', 'elevation', 'management', 'level_of_diagnostic_difficulty']
meta_out = df[[c for c in keep if c in df.columns]].copy()
meta_out['split'] = split_arr
for i, name in enumerate(concept_names):
    meta_out['gt_' + name] = gt_concepts[:, i]
meta_out['gt_7pt_score'] = gt_score
try:
    meta_out.to_parquet(os.path.join(OUT_DIR, 'meta.parquet'))
except Exception as e:
    print('parquet failed, csv fallback:', e)
    meta_out.to_csv(os.path.join(OUT_DIR, 'meta.csv'), index=False)

manifest = {
    'model': MODEL_NAME,
    'embedding_dim': int(emb_derm.shape[1]),
    'n_cases': int(len(df)),
    'concept_names': concept_names,
    'concept_weights': WEIGHTS.tolist(),
    'splits': {sn: int((split_arr == sn).sum()) for sn in ['train', 'val', 'test', 'unused']},
    'created': time.strftime('%Y-%m-%d %H:%M:%S'),
    'notes': 'DermLIP ViT-B/16 frozen embeddings + 7pt concept scores for Derm7pt. Input to Notebook B (policy-gradient + faithfulness reward).',
}
with open(os.path.join(OUT_DIR, 'manifest.json'), 'w') as fh:
    json.dump(manifest, fh, indent=2)

print('Saved cache to', OUT_DIR)
print(os.listdir(OUT_DIR))

## What you just produced

`features.npz` holds, per case: frozen DermLIP image embeddings (dermoscopic + clinical), 7 DermLIP concept-presence scores, the Derm7pt ground-truth concept labels, the rule-based 7-point score, melanoma label, and the split. This is the entire substrate for Notebook B.

**Read the sanity output before moving on:**
- Zero-shot melanoma AUROC well above 0.5 -> DermLIP loaded correctly.
- Most per-concept AUROCs above ~0.6 -> the text prompts genuinely capture the clinical criteria (your concept layer is real, not noise). Concepts that score near 0.5 are candidates to re-prompt or to supervise with a probe in Notebook B.

**Publish for reuse:** Save Version (Save & Run All) -> on the output, *Create Dataset* -> attach that dataset as input to Notebook B so you never recompute embeddings.

Next: **Notebook B** trains a small concept->diagnosis policy on these cached vectors with the composite reward (correctness + concept faithfulness via the 7-point rule) using single-step policy gradient, and runs the intervention-consistency test (hypothesis H1).